In [1]:
import os
import sys

import matplotlib
matplotlib.use("Qt5Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pypsa

In [40]:
new_bus = "New Battery Node"
scenario_paths = ("networks/SV2024_north-west.nc", "networks/WP2033_north-west.nc")
chosen = 1
x, y = -8.5, 54.5  # lon, lat (deg)
new_bus_v_nom = 110

# CHANGED: sweep of K values to test, instead of a single hardcoded K = 3
k_values = [1, 2, 3, 4, 5]
results = [] 


def total_curtailment_mwh(network, renewable_carriers=("wind",)):
    gens = network.generators
    curtailable = gens[gens.carrier.isin(renewable_carriers)].index

    available = (
        network.generators_t.p_max_pu[curtailable]
        * gens.loc[curtailable, "p_nom_opt"]
    )
    dispatched = network.generators_t.p[curtailable]

    curtailed_mw = (available - dispatched).clip(lower=0)
    weighted = curtailed_mw.mul(network.snapshot_weightings.generators, axis=0)
    return weighted.sum().sum()


def haversine_km(lon0, lat0, lon1, lat1):
    """Return great-circle distance in kilometres."""
    R = 6371.0
    p0, p1 = np.radians(lat0), np.radians(lat1)
    dphi = np.radians(lat1 - lat0)
    dlambda = np.radians(lon1 - lon0)
    a = np.sin(dphi / 2)**2 + np.cos(p0) * np.cos(p1) * np.sin(dlambda / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def surplus_waste_reduction_pct(network, curtailment_before, curtailment_after,
                                  renewable_carriers=("wind",)):
    gens = network.generators
    curtailable = gens[gens.carrier.isin(renewable_carriers)].index
    available = (
        network.generators_t.p_max_pu[curtailable]
        * gens.loc[curtailable, "p_nom_opt"]
    )
    total_available_mwh = available.mul(
        network.snapshot_weightings.generators, axis=0
    ).sum().sum()

    avoided_mwh = curtailment_before - curtailment_after
    return avoided_mwh / total_available_mwh * 100


n_baseline = pypsa.Network(scenario_paths[chosen])
n_baseline.optimize()
curtailment_before = total_curtailment_mwh(n_baseline)

n = pypsa.Network(scenario_paths[chosen])
n.snapshot_weightings["objective"] *= 8760 / 168

curtailable = n.generators.index[n.generators.carrier == "wind"]
curtailment_penalty = 100
n.generators.loc[curtailable, "marginal_cost"] -= curtailment_penalty

n.add("Bus", new_bus, x=x, y=y, v_nom=new_bus_v_nom)

n.add("StorageUnit",
      f"Battery at {new_bus}",
      bus=new_bus,
      p_nom_extendable=True,
      p_nom_min=0,
      p_nom_max=2_000,
      capital_cost=75_000,
      marginal_cost=0.1,
      efficiency_store=0.95,
      efficiency_dispatch=0.95,
      max_hours=4,
      cyclic_state_of_charge=True)

line_cost_per_mw_km = 300
x_per_km = 0.35  # ohm/km
r_per_km = 0.12  # ohm/km

candidate_buses = [
    bus for bus in n.buses.index
    if bus != new_bus and n.buses.at[bus, "v_nom"] == new_bus_v_nom
]
skipped = set(n.buses.index) - set(candidate_buses) - {new_bus}
if skipped:
    print(f"Skipping {len(skipped)} buses at a different voltage "
          f"(would need a Transformer, not a Line): {sorted(skipped)[:5]}...")

for bus in candidate_buses:
    x0, y0 = n.buses.at[bus, "x"], n.buses.at[bus, "y"]
    length = haversine_km(x, y, x0, y0)
    n.add("Line",
          f"{new_bus} - {bus}",
          bus0=new_bus, bus1=bus,
          x=x_per_km * length,
          r=r_per_km * length,
          s_nom_extendable=True,
          s_nom_min=0,
          s_nom_max=200,
          capital_cost=length * line_cost_per_mw_km,
          length=length)

# This first-pass optimize only needs to run once — it doesn't depend on K,
# it just ranks candidate buses by how much line capacity gets built to them.
n.optimize()

curtailment_after_lines_only = total_curtailment_mwh(n)
print(f"Curtailment after lines alone: {curtailment_after_lines_only:.2f} MWh "
      f"({(curtailment_before - curtailment_after_lines_only) / curtailment_before * 100:.1f}% reduction)")

built = {}
for line_name in n.lines.index:
    if line_name.startswith(new_bus):
        s_opt = n.lines.at[line_name, "s_nom_opt"]
        if s_opt > 0.01:
            existing_bus = line_name.split(" - ")[1]
            built[existing_bus] = s_opt

ranked_buses = sorted(built, key=built.get, reverse=True)

first_pass_lines = n.lines.index[n.lines.index.str.startswith(new_bus)]
n.remove("Line", first_pass_lines)


n.model.solver_model = None
n_clean = n.copy()

min_line_cap = 0

for K in k_values:
    n.model.solver_model = None
    n_k = n_clean.copy()
    top_buses = ranked_buses[:K]

    for bus in top_buses:
        x0, y0 = n_k.buses.at[bus, "x"], n_k.buses.at[bus, "y"]
        length = haversine_km(x, y, x0, y0)
        n_k.add("Line",
                f"{new_bus} - {bus}",
                bus0=new_bus, bus1=bus,
                x=x_per_km * length,
                r=r_per_km * length,
                s_nom_extendable=True,
                s_nom_min=min_line_cap,
                s_nom_max=200,
                capital_cost=length * line_cost_per_mw_km,
                length=length)

    n_k.optimize()

    final_lines = {}
    for line_name in n_k.lines.index:
        if line_name.startswith(new_bus):
            s_opt = n_k.lines.at[line_name, "s_nom_opt"]
            if s_opt > 0.1:
                final_lines[line_name] = s_opt

    battery_opt = n_k.storage_units.at[f"Battery at {new_bus}", "p_nom_opt"]
    curtailment_after = total_curtailment_mwh(n_k)

    if curtailment_before > 0:
        pct_change = (curtailment_after - curtailment_before) / curtailment_before * 100
    else:
        pct_change = float("nan")

    waste_reduction = surplus_waste_reduction_pct(n_k, curtailment_before, curtailment_after)

    # CHANGED: store this K's results instead of just printing them
    results.append({
        "K": K,
        "top_buses": top_buses,
        "built_lines": final_lines,
        "battery_mw": battery_opt,
        "battery_mwh": battery_opt * 4,
        "curtailment_before_mwh": curtailment_before,
        "curtailment_after_mwh": curtailment_after,
        "dispatch_down_pct_change": pct_change,
        "surplus_waste_reduction_pct": waste_reduction,
        "network": n_k,  # NEW: keep the solved network so we can plot the best one later
    })

# CHANGED: display all K results together at the end, instead of printing inline per-run
print(f"\n{'K':>3} | {'Battery (MW)':>13} | {'Battery (MWh)':>14} | {'Curtailment after (MWh)':>24} | {'Dispatch-down Δ%':>17} | {'Waste reduced %':>16}")
print("-" * 100)
for r in results:
    print(f"{r['K']:>3} | {r['battery_mw']:>13.2f} | {r['battery_mwh']:>14.2f} | "
          f"{r['curtailment_after_mwh']:>24.2f} | {r['dispatch_down_pct_change']:>17.1f} | "
          f"{r['surplus_waste_reduction_pct']:>16.2f}")


# Optimal K = the one that reduces curtailment the most (highest waste-reduction %)
best_result = max(results, key=lambda r: r["surplus_waste_reduction_pct"])
best_K = best_result["K"]
n_best = best_result["network"]

print(f"\nOptimal K = {best_K} "
      f"(waste reduction: {best_result['surplus_waste_reduction_pct']:.2f}%, "
      f"battery: {best_result['battery_mw']:.2f} MW)")


INFO:pypsa.network.io:Imported network 'TYTFS2024_WP2033_V35 north-west (aggregated)' has buses, carriers, generators, lines, loads
/tmp/ipykernel_5347/1466803628.py:54: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n_baseline.optimize()
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 8904 primals, 21168 duals
Objective: -3.94e+06
Solver: highs
Runtime: 0.03s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Kirchhoff-Voltage-Law 

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-ozjs_pw3 has 21168 rows; 8904 cols; 32088 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 2e+02]
  Cost    [1e+00, 1e+04]
  Bound   [0e+00, 0e+00]
  RHS     [4e-01, 5e+02]
Presolving model
2012 rows, 5706 cols, 8888 nonzeros 0s
1254 rows, 4942 cols, 7548 nonzeros 0s
978 rows, 4041 cols, 6529 nonzeros 0s
978 rows, 3175 cols, 5564 nonzeros 0s
Dependent equations search running on 978 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
978 rows, 3175 cols, 5564 nonzeros 0s
Presolve reductions: rows 978(-20190); columns 3175(-5729); nonzeros 5564(-26524) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
       1134    -3.93512440

/tmp/ipykernel_5347/1466803628.py:108: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize()
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 7/7 [00:00<00:00, 570.42it/s]
INFO:linopy.io: Writing time: 0.08s


Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-363souf8 has 29262 rows; 11775 cols; 54798 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 3e+02]
  Cost    [5e+00, 5e+05]
  Bound   [0e+00, 0e+00]
  RHS     [4e-01, 2e+03]
Presolving model
10416 rows, 9759 cols, 32928 nonzeros 0s
8484 rows, 7827 cols, 34732 nonzeros 0s
Dependent equations search running on 2626 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
7834 rows, 7177 cols, 37333 nonzeros 0s
Presolve reductions: rows 7834(-21428); columns 7177(-4598); nonzeros 37333(-17465) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -2.3096511427e-08 Ph1: 4917(4.50025e+06); Du: 0(2.63974e-10) 0.0s
       3361    -6.0242679214e+08 Pr: 0(0); Du: 0(1.53796e-10

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 11775 primals, 29262 duals
Objective: -6.02e+08
Solver: highs
Runtime: 0.19s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Line-ext-s-lower, Line-ext-s-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, Kirchhoff-Voltage-Law, StorageUnit-energy_balance were not assigned to the network.
/tmp/ipykernel_5347/1466803628.py:152: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical cond

...

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-5hs_6kck has 23696 rows; 9916 cols; 38312 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 2e+02]
  Cost    [5e+00, 5e+05]
  Bound   [0e+00, 0e+00]
  RHS     [4e-01, 2e+03]
Presolving model
4368 rows, 7396 cols, 15456 nonzeros 0s
3272 rows, 6300 cols, 14952 nonzeros 0s
3058 rows, 5078 cols, 14119 nonzeros 0s
Dependent equations search running on 1546 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
3058 rows, 5078 cols, 14119 nonzeros 0s
Presolve reductions: rows 3058(-20638); columns 5078(-4838); nonzeros 14119(-24193) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
       2000    -5.8231760242e+08 Pr: 0(0); Du: 0(1.4562

/tmp/ipykernel_5347/1466803628.py:152: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n_k.optimize()
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 7/7 [00:00<00:00, 695.10it/s]
INFO:linopy.io: Writing time: 0.07s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 10085 primals, 24202 duals
Objective: -5.99e+08
Solver: highs
Runtime: 0.08s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Line-ext-s-lower, Line-ext-s-upper, StorageUnit

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-dcqubt9_ has 24202 rows; 10085 cols; 39994 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 2e+02]
  Cost    [5e+00, 5e+05]
  Bound   [0e+00, 0e+00]
  RHS     [4e-01, 2e+03]
Presolving model
4872 rows, 7565 cols, 17136 nonzeros 0s
3608 rows, 6301 cols, 16800 nonzeros 0s
Dependent equations search running on 1524 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
3372 rows, 5057 cols, 16263 nonzeros 0s
Presolve reductions: rows 3372(-20830); columns 5057(-5028); nonzeros 16263(-23731) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
       2077    -5.9909240620e+08 Pr: 0(0); Du: 0(1.08367e-10) 0.1s

Performed postsolve
Solvin

/tmp/ipykernel_5347/1466803628.py:152: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n_k.optimize()
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 7/7 [00:00<00:00, 779.42it/s]
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 10254 primals, 24708 duals
Objective: -6.04e+08
Solver: highs
Runtime: 0.09s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Line-ext-s-lower, Line-ext-s-upper, StorageUnit

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-u74mfkvk has 24708 rows; 10254 cols; 41676 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 2e+02]
  Cost    [5e+00, 5e+05]
  Bound   [0e+00, 0e+00]
  RHS     [4e-01, 2e+03]
Presolving model
5544 rows, 7902 cols, 19152 nonzeros 0s
4112 rows, 6470 cols, 19684 nonzeros 0s
Dependent equations search running on 1870 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
4054 rows, 5572 cols, 19240 nonzeros 0s
Presolve reductions: rows 4054(-20654); columns 5572(-4682); nonzeros 19240(-22436) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
       2410    -6.0437248408e+08 Pr: 0(0); Du: 0(5.11348e-10) 0.1s

Performed postsolve
Solvin

In [41]:
# detach the solved model before plotting, in case you want to .copy() n_best later
n_best.model.solver_model = None

bus_colours = pd.Series("lightgray", index=n_best.buses.index, dtype="object")

wind_buses = n_best.generators.loc[n_best.generators.carrier == "wind", "bus"]
other_generator_buses = n_best.generators.loc[n_best.generators.carrier != "wind", "bus"]
load_buses = n_best.loads["bus"]
battery_buses = n_best.storage_units["bus"]

bus_colours.loc[other_generator_buses.unique()] = "red"
bus_colours.loc[wind_buses.unique()] = "green"
bus_colours.loc[load_buses.unique()] = "blue"
bus_colours.loc[battery_buses.unique()] = "orange"

n_best.plot(bus_sizes=0.0025, margin=0.25, bus_colors=bus_colours)
plt.title(f"Optimal network topology (K={best_K})")
plt.show()

qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
